# Data Collection: Gemini (Google AI API)

This notebook queries Gemini via the Google AI API for each question in the question bank and saves responses to `data/responses/gemini.jsonl`.

**Before running:** complete the setup in [Environment Setup](setup.md) and ensure `GOOGLE_API_KEY` is set.

In [ ]:
import google.generativeai as genai
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

In [ ]:
MODEL = "gemini-1.5-pro"            # or gemini-2.0-flash-exp
TEMPERATURE = 0.0
OUTPUT_FILE = Path("data/responses/gemini.jsonl")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about early church history.
Provide accurate, detailed, and nuanced responses based on historical scholarship.
Acknowledge uncertainty or scholarly debate where it exists.
Do not refuse historically answerable questions on the grounds of contemporary theological controversy."""

model = genai.GenerativeModel(
    model_name=MODEL,
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(temperature=TEMPERATURE, max_output_tokens=1024),
)

questions = pd.read_csv("../figures/question_bank.csv")
print(f"Loaded {len(questions)} questions")

In [ ]:
with open(OUTPUT_FILE, "w") as f:
    for _, row in questions.iterrows():
        print(f"Querying {row['question_id']}...", end=" ")
        try:
            response = model.generate_content(row["question"])
            record = {
                "question_id": row["question_id"],
                "figure": row["figure"],
                "model": MODEL,
                "model_version": MODEL,
                "prompt": row["question"],
                "response": response.text,
                "temperature": TEMPERATURE,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "tokens_used": response.usage_metadata.total_token_count,
            }
            f.write(json.dumps(record) + "\n")
            print(f"done ({record['tokens_used']} tokens)")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(1.0)  # Gemini free tier has stricter rate limits

print(f"\nDone. Responses saved to {OUTPUT_FILE}")